In [1]:
!pip install kagglehub

In [4]:
import kagglehub
import os

path = kagglehub.dataset_download("hijest/genre-classification-dataset-imdb")

print("Dataset path:", path)

print(os.listdir(path))

Using Colab cache for faster access to the 'genre-classification-dataset-imdb' dataset.
Dataset path: /kaggle/input/genre-classification-dataset-imdb
['.nfs000000008e6821fe00000470', 'Genre Classification Dataset']


In [18]:

import os
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report


# TRAIN DATA
train_file = os.path.join(path, "/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/train_data.txt")

with open(train_file, "r", encoding="utf-8") as f:
    train_lines = f.readlines()

train_data = []

for line in train_lines:
    parts = [p.strip() for p in line.strip().split(":::")]

    if len(parts) >= 4:
        _, title, genre, description = parts[:4]
        train_data.append([description, genre])

df_train = pd.DataFrame(train_data, columns=["plot", "genre"])

# Clean empty plots
df_train = df_train[df_train["plot"].str.strip() != ""]

print("Train shape:", df_train.shape)


# TEST DATA
test_file = os.path.join(path, "/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data.txt")

with open(test_file, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

test_data = []

for line in test_lines:
    parts = [p.strip() for p in line.strip().split(":::")]

    if len(parts) >= 3:
        _, title, description = parts[:3]
        test_data.append(description)

df_test = pd.DataFrame(test_data, columns=["plot"])

# Clean empty plots
df_test = df_test[df_test["plot"].str.strip() != ""]

print("Test shape:", df_test.shape)


# SOLUTION FILE
solution_file = os.path.join(path, "/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data_solution.txt")

with open(solution_file, "r", encoding="utf-8") as f:
    solution_lines = f.readlines()

solution_data = []

for line in solution_lines:
    parts = [p.strip() for p in line.strip().split(":::")]

    if len(parts) >= 4:
        _, title, genre, description = parts[:4]
        solution_data.append(genre)

df_solution = pd.DataFrame(solution_data, columns=["genre"])

print("Solution shape:", df_solution.shape)


# Ensure alignment
min_len = min(len(df_test), len(df_solution))
df_test = df_test.iloc[:min_len]
df_solution = df_solution.iloc[:min_len]

print("Final aligned size:", len(df_test), len(df_solution))


# Define Train/Test
X_train = df_train["plot"]
y_train = df_train["genre"]

X_test = df_test["plot"]
y_test = df_solution["genre"]


# Build Model (TF-IDF + SVM)
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1,2)
    )),
    ("clf", LinearSVC(class_weight="balanced"))
])


# Train Model
model.fit(X_train, y_train)


# Predict
y_pred = model.predict(X_test)


# Evaluate
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Train shape: (54214, 2)
Test shape: (54200, 1)
Solution shape: (54200, 1)
Final aligned size: 54200 54200

Classification Report:

              precision    recall  f1-score   support

      action       0.27      0.41      0.33      1314
       adult       0.30      0.54      0.38       590
   adventure       0.17      0.28      0.21       775
   animation       0.13      0.23      0.17       498
   biography       0.03      0.08      0.05       264
      comedy       0.59      0.44      0.50      7446
       crime       0.11      0.25      0.15       505
 documentary       0.77      0.67      0.71     13096
       drama       0.68      0.43      0.53     13612
      family       0.12      0.25      0.17       783
     fantasy       0.09      0.20      0.12       322
   game-show       0.54      0.67      0.60       193
     history       0.05      0.10      0.07       243
      horror       0.49      0.62      0.55      2204
       music       0.39      0.62      0.48       731
    